In [1]:
import pandas as pd

df = pd.read_csv(
    "../data/processed/us_gun_policy_supervised_v3.csv"
)

treino = df[df["year"] <= 2022].copy()
teste = df[df["year"] == 2023].copy()

print("Treino:", treino.shape)
print("Teste:", teste.shape)

print("\nAnos de treinamento:", sorted(treino["year"].unique()))
print("Anos de teste:", sorted(teste["year"].unique()))

Treino: (200, 13)
Teste: (50, 13)

Anos de treinamento: [np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022)]
Anos de teste: [np.int64(2023)]


In [2]:
variaveis_a = [
    "homicide_rate",
    "unemployment_rate",
    "income_per_capita"
]

variaveis_legislacao = [
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited",
    "background_handgun",
    "background_longgun"
]

variaveis_b = variaveis_a + variaveis_legislacao

alvo = "homicide_rate_next_year"

print("Modelo A:", variaveis_a)
print("\nModelo B:", variaveis_b)
print("\nVariável-alvo:", alvo)

print("\nIndicadores legislativos por ano:")
display(
    treino.groupby("year")[variaveis_legislacao].sum()
)

Modelo A: ['homicide_rate', 'unemployment_rate', 'income_per_capita']

Modelo B: ['homicide_rate', 'unemployment_rate', 'income_per_capita', 'ccw_shall_issue', 'ccw_permitless', 'ccw_may_issue', 'ccw_prohibited', 'background_handgun', 'background_longgun']

Variável-alvo: homicide_rate_next_year

Indicadores legislativos por ano:


,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited,background_handgun,background_longgun
year,,,,,,
2019,27,15,8,0,18,15
2020,27,15,8,0,19,16
2021,21,21,8,0,19,16
2022,24,24,2,0,18,15


In [3]:
colunas_ccw = [
    "ccw_shall_issue",
    "ccw_permitless",
    "ccw_may_issue",
    "ccw_prohibited"
]

df["total_ccw"] = df[colunas_ccw].sum(axis=1)

print("Distribuição das modalidades por estado-ano:")
display(df["total_ccw"].value_counts().sort_index())

print("\nEstados com mais de uma modalidade:")
display(
    df.loc[
        df["total_ccw"] > 1,
        ["state", "year"] + colunas_ccw
    ]
)

Distribuição das modalidades por estado-ano:


total_ccw
1    250
Name: count, dtype: int64


Estados com mais de uma modalidade:


,state,year,ccw_shall_issue,ccw_permitless,ccw_may_issue,ccw_prohibited


In [4]:
ccw = df.sort_values(["state", "year"]).copy()

ccw["modalidade"] = ccw[colunas_ccw].idxmax(axis=1)

ccw["modalidade_anterior"] = (
    ccw.groupby("state")["modalidade"].shift(1)
)

mudancas = ccw[
    ccw["modalidade_anterior"].notna()
    & (ccw["modalidade"] != ccw["modalidade_anterior"])
]

display(
    mudancas[
        ["state", "year", "modalidade_anterior", "modalidade"]
    ].sort_values(["year", "state"])
)

print("Total de mudanças:", len(mudancas))

,state,year,modalidade_anterior,modalidade
17,Arkansas,2021,ccw_shall_issue,ccw_permitless
72,Iowa,2021,ccw_shall_issue,ccw_permitless
127,Montana,2021,ccw_shall_issue,ccw_permitless
207,Tennessee,2021,ccw_shall_issue,ccw_permitless
212,Texas,2021,ccw_shall_issue,ccw_permitless
217,Utah,2021,ccw_shall_issue,ccw_permitless
23,California,2022,ccw_may_issue,ccw_shall_issue
48,Georgia,2022,ccw_shall_issue,ccw_permitless
53,Hawaii,2022,ccw_may_issue,ccw_shall_issue
68,Indiana,2022,ccw_shall_issue,ccw_permitless


Total de mudanças: 18


In [5]:
import pandas as pd

arquivo_rand = (
    "../reports/attachments/"
    "TL-A243-2-v4 State Firearm Law Database 6.0.xlsx"
)

rand = pd.read_excel(
    arquivo_rand,
    sheet_name="Database"
)

print(rand.columns.tolist())

['Law ID', 'State', 'State Postal Abbreviation', 'FIPS Code', 'Law Class (num)', 'Law Class', 'Law Class Subtype', 'Handguns or Long Guns', 'Effect', 'Type of Change', 'Effective Date Note', 'Effective Date Year', 'Effective Date Month', 'Effective Date Day', 'Statutory Citation', 'Content', 'Supersession Date Year', 'Supersession Date Month', 'Supersession Date Day', 'Age for Minimum Age Laws', 'Length of Waiting Period (days, handguns)', 'Additional Context and Notes', 'Discrepancies']


In [6]:
estados_bruen = [
    "California",
    "Hawaii",
    "Maryland",
    "Massachusetts",
    "New Jersey",
    "New York"
]

colunas_inspecao = [
    "Law ID",
    "State",
    "Law Class Subtype",
    "Effect",
    "Type of Change",
    "Effective Date Note",
    "Effective Date Year",
    "Effective Date Month",
    "Effective Date Day",
    "Supersession Date Year",
    "Additional Context and Notes"
]

rand_bruen = rand[
    rand["State"].isin(estados_bruen)
    & rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
    & rand["Effective Date Year"].between(2021, 2023)
]

print("Registros encontrados:", len(rand_bruen))

display(rand_bruen[colunas_inspecao])

Registros encontrados: 7


,Law ID,State,Law Class Subtype,Effect,Type of Change,Effective Date Note,Effective Date Year,Effective Date Month,Effective Date Day,Supersession Date Year,Additional Context and Notes
178,CA1107,California,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,NaN
522,HI1084,Hawaii,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
823,MA1068,Massachusetts,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
872,MD1065,Maryland,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
1236,NJ1072,New Jersey,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...
1248,NJ1085,New Jersey,shall issue,Permissive,Modify,NaN,2022,12,22,NaN,NaN
1350,NY1069,New York,shall issue,Permissive,Modify,NaN,2022,6,23,NaN,The Supreme court decision in Bruen determined...


In [7]:
anteriores = rand[
    rand["State"].isin(estados_bruen)
    & rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
    & rand["Law Class Subtype"].astype(str).str.contains(
        "may issue",
        case=False,
        na=False
    )
]

display(
    anteriores[
        [
            "Law ID",
            "State",
            "Law Class Subtype",
            "Effective Date Year",
            "Supersession Date Year",
            "Supersession Date Month",
            "Supersession Date Day"
        ]
    ]
)

,Law ID,State,Law Class Subtype,Effective Date Year,Supersession Date Year,Supersession Date Month,Supersession Date Day
112,CA1007,California,may issue,1953,2022.0,6.0,23.0
470,HI1005,Hawaii,may issue,1961,2022.0,6.0,23.0
781,MA1004,Massachusetts,may issue,1906,2022.0,6.0,23.0
840,MD1005,Maryland,may issue,1972,2022.0,6.0,23.0
1194,NJ1006,New Jersey,may issue,1924,2022.0,6.0,23.0
1315,NY1008,New York,may issue,1973,2022.0,6.0,23.0


In [8]:
estados_permitless = [
    "Arkansas", "Iowa", "Montana", "Tennessee",
    "Texas", "Utah", "Georgia", "Indiana",
    "Ohio", "Alabama", "Florida", "Nebraska"
]

registros_permitless = rand[
    rand["State"].isin(estados_permitless)
    & rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
    & rand["Law Class Subtype"].astype(str).str.contains(
        "permitless",
        case=False,
        na=False
    )
    & rand["Effective Date Year"].between(2020, 2023)
]

display(
    registros_permitless[
        [
            "Law ID",
            "State",
            "Law Class Subtype",
            "Type of Change",
            "Effective Date Year",
            "Effective Date Month",
            "Effective Date Day"
        ]
    ].sort_values(["Effective Date Year", "State"])
)

print("Registros encontrados:", len(registros_permitless))

,Law ID,State,Law Class Subtype,Type of Change,Effective Date Year,Effective Date Month,Effective Date Day


Registros encontrados: 0


In [9]:
ccw_rand = rand[
    rand["Law Class"].astype(str).str.contains(
        "carrying a concealed weapon",
        case=False,
        na=False
    )
].copy()

print("Registros CCW:", len(ccw_rand))

print("\nSubcategorias disponíveis:")
display(ccw_rand["Law Class Subtype"].value_counts(dropna=False))

print("\nEfeitos disponíveis:")
display(ccw_rand["Effect"].value_counts(dropna=False))

Registros CCW: 142

Subcategorias disponíveis:


Law Class Subtype
shall issue                          56
may issue                            35
shall issue (permit not required)    29
prohibited                           22
Name: count, dtype: int64


Efeitos disponíveis:


Effect
Permissive     117
Restrictive     25
Name: count, dtype: int64